# 🎵 Spotify Tracks — Phân Tích Khám Phá Dữ Liệu (EDA) & Dự Đoán Mức Độ Phổ Biến

> **Quy trình phân tích đầy đủ**: Làm sạch dữ liệu → Trực quan hóa → Feature Engineering → Huấn luyện & Đánh giá mô hình

---
**Bộ dữ liệu**: ~114.000 bài hát Spotify thuộc 114 thể loại  
**Mục tiêu**: Khám phá yếu tố nào khiến một bài hát trở nên phổ biến + Xây dựng mô hình hồi quy dự đoán độ phổ biến  

---

## 📦 1. Import Thư Viện & Cấu Hình

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb

# ── Giao diện Spotify Dark Theme ─────────────────────────────────────────────
SPOTIFY_GREEN  = '#1DB954'
SPOTIFY_BLACK  = '#191414'
SPOTIFY_WHITE  = '#FFFFFF'
ACCENT_PURPLE  = '#9B59B6'
ACCENT_ORANGE  = '#E67E22'
ACCENT_BLUE    = '#2E86AB'
PALETTE        = [SPOTIFY_GREEN, ACCENT_PURPLE, ACCENT_ORANGE, ACCENT_BLUE,
                  '#E74C3C', '#F1C40F', '#1ABC9C', '#E91E63']

plt.rcParams.update({
    'figure.facecolor' : SPOTIFY_BLACK,
    'axes.facecolor'   : '#121212',
    'axes.edgecolor'   : '#333333',
    'axes.labelcolor'  : SPOTIFY_WHITE,
    'xtick.color'      : '#AAAAAA',
    'ytick.color'      : '#AAAAAA',
    'text.color'       : SPOTIFY_WHITE,
    'grid.color'       : '#2A2A2A',
    'grid.linestyle'   : '--',
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 12,
})

print('✅ Đã tải thư viện — Giao diện Spotify dark theme được kích hoạt')

## 📂 2. Đọc & Làm Sạch Dữ Liệu

In [ ]:
# Đọc dữ liệu từ file CSV
df = pd.read_csv('../dataset.csv')
print(f'Kích thước ban đầu: {df.shape}')
df.head()

In [ ]:
# === Làm sạch dữ liệu (giống train_model.py) ===

# Xóa cột index không cần thiết
df.drop(columns=[c for c in df.columns if 'Unnamed' in c], inplace=True, errors='ignore')

# Loại bỏ các bài hát trùng lặp (cùng track_id)
df.drop_duplicates(subset='track_id', inplace=True)
df.reset_index(drop=True, inplace=True)

# Chuyển đổi thời lượng từ mili-giây sang phút
df['duration_min'] = df['duration_ms'] / 60_000

# Chuyển cột explicit sang kiểu số (0/1)
df['explicit'] = df['explicit'].astype(int)

print(f'Kích thước sau khi làm sạch: {df.shape}')
print(f'\nGiá trị bị thiếu (missing values):')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Không có giá trị bị thiếu!')
df.head(3)

### 📝 Nhận xét về dữ liệu sau khi làm sạch

- **Kích thước dữ liệu**: Sau khi loại bỏ các bản ghi trùng lặp dựa trên `track_id`, bộ dữ liệu giảm từ ~114.000 xuống ~89.741 bài hát. Điều này cho thấy có **khoảng 21% bài hát bị lặp** trong dataset gốc — một tỷ lệ đáng kể cần được xử lý.
- **Giá trị thiếu**: Chỉ có vài giá trị thiếu ở các cột `artists`, `album_name`, `track_name` — không ảnh hưởng đáng kể đến phân tích vì các cột này không được sử dụng làm features cho mô hình.
- **Chuyển đổi đơn vị**: Cột `duration_ms` (mili-giây) được chuyển sang `duration_min` (phút) để dễ hiểu hơn khi trực quan hóa.
- **Biến explicit**: Được chuyển từ `True/False` sang `0/1` để phục vụ cho mô hình học máy.
- **Quy trình hoàn toàn giống `train_model.py`** — đảm bảo tính nhất quán giữa EDA và pipeline huấn luyện.

In [ ]:
# Thống kê mô tả nhanh
df.describe().T.style.background_gradient(cmap='Greens')

### 📝 Nhận xét bảng thống kê mô tả

- **Popularity (Mức độ phổ biến)**: Trung bình chỉ khoảng **33/100**, trung vị cũng là 33. Phân bố này cho thấy đa số bài hát trên Spotify có mức độ phổ biến **thấp đến trung bình**. Có bài hát đạt popularity = 0 (hoàn toàn không phổ biến) và 100 (cực kỳ viral).
- **Duration (Thời lượng)**: Trung bình ~3.8 phút, nhưng max lên tới ~87 phút — cho thấy có **outlier** (bài hát dài bất thường, có thể là podcast hoặc mix).
- **Explicit**: Chỉ ~8.6% bài hát có nội dung người lớn — phần lớn nhạc trên Spotify **không explicit**.
- **Danceability**: Trung bình 0.56, phân bố khá đều — cho thấy bộ dữ liệu **đa dạng về khả năng nhảy**.
- **Energy**: Trung bình 0.63, nghiêng về phía năng lượng cao.
- **Loudness**: Trung bình -8.5 dB, phạm vi rộng từ -49.5 đến 4.5 dB.
- **Instrumentalness**: Trung bình 0.17 nhưng trung vị gần 0 — hầu hết bài hát **có lời hát**, chỉ một phần nhỏ là nhạc không lời.
- **Tempo**: Trung bình ~122 BPM, nằm trong khoảng nhạc pop/dance điển hình.

---
## 📊 3. Phân Tích Khám Phá Dữ Liệu (EDA)
### 3.1 Phân Phối Mức Độ Phổ Biến (Popularity Distribution)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('🎵  Phân Phối Mức Độ Phổ Biến', fontsize=18, fontweight='bold',
             color=SPOTIFY_GREEN, y=1.02)

# Histogram
ax = axes[0]
ax.hist(df['popularity'], bins=50, color=SPOTIFY_GREEN, edgecolor='#000',
        linewidth=0.4, alpha=0.9)
ax.axvline(df['popularity'].mean(), color=ACCENT_ORANGE, linestyle='--',
           lw=2, label=f'Trung bình = {df["popularity"].mean():.1f}')
ax.axvline(df['popularity'].median(), color=ACCENT_PURPLE, linestyle='-.',
           lw=2, label=f'Trung vị = {df["popularity"].median():.0f}')
ax.set_xlabel('Popularity'); ax.set_ylabel('Số lượng bài hát')
ax.set_title('Phân phối tần suất (Histogram)', fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.2)

# Box plot
ax = axes[1]
bp = ax.boxplot(df['popularity'], vert=False, patch_artist=True, widths=0.6,
                boxprops=dict(facecolor=SPOTIFY_GREEN, alpha=0.7),
                medianprops=dict(color='white', lw=2.5),
                flierprops=dict(marker='.', markerfacecolor='#888', markersize=2))
ax.set_xlabel('Popularity')
ax.set_title('Biểu đồ hộp (Box Plot)', fontweight='bold')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

### 📝 Nhận xét biểu đồ 3.1 — Phân Phối Mức Độ Phổ Biến

#### Histogram (Biểu đồ phân phối tần suất):
- Phân phối popularity **không tuân theo phân phối chuẩn** (normal distribution), mà có dạng **lệch phải** (right-skewed) kết hợp với một **đỉnh rõ rệt tại giá trị 0**.
- Có một lượng lớn bài hát có popularity = 0, cho thấy nhiều bài hát trên Spotify **hầu như không có lượt nghe**. Đây có thể là các bài hát mới tải lên, bài hát từ nghệ sĩ ít tên tuổi, hoặc bài hát cũ không còn được nghe.
- **Trung bình (~33)** và **trung vị (~33)** gần bằng nhau, nhưng do đỉnh lớn ở giá trị 0, phân phối thực tế **không đối xứng**.
- Phần lớn bài hát tập trung trong khoảng **15-55** popularity, rất ít bài hát đạt trên 80.
- Điều này phản ánh quy luật thực tế: **chỉ một tỷ lệ nhỏ bài hát trở nên cực kỳ phổ biến** (viral), còn đa số nằm ở mức trung bình hoặc thấp.

#### Box Plot (Biểu đồ hộp):
- Hộp (IQR) trải từ khoảng **19 đến 49**, cho thấy 50% bài hát nằm trong khoảng phổ biến trung bình.
- **Không có outlier rõ rệt ở phía trên** — popularity = 100 vẫn nằm trong phạm vi hợp lý.

#### Ý nghĩa cho bài toán dự đoán:
- Biến mục tiêu `popularity` có phân phối **không đều**, điều này có thể gây khó khăn cho các mô hình hồi quy tuyến tính (Ridge).
- Nên cân nhắc sử dụng các mô hình **phi tuyến** (Random Forest, XGBoost, LightGBM) — đúng như pipeline trong `train_model.py`.

### 3.2 Ma Trận Tương Quan Các Đặc Trưng Âm Thanh (Audio Feature Correlations)

In [ ]:
# Sử dụng đúng các features trong project (numeric_features + popularity)
AUDIO_FEATURES = ['popularity', 'danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

corr = df[AUDIO_FEATURES].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))

cmap = sns.diverging_palette(145, 300, s=80, l=55, as_cmap=True)  # xanh lá ↔ tím
sns.heatmap(corr, mask=mask, cmap=cmap, center=0, annot=True, fmt='.2f',
            linewidths=.5, linecolor='#333', cbar_kws={'shrink': 0.8},
            annot_kws={'size': 11, 'weight': 'bold'})

ax.set_title('🔗  Ma Trận Tương Quan — Các Đặc Trưng Âm Thanh',
             fontsize=16, fontweight='bold', color=SPOTIFY_GREEN, pad=15)
plt.tight_layout()
plt.show()

### 📝 Nhận xét biểu đồ 3.2 — Ma Trận Tương Quan

#### Các cặp tương quan mạnh nhất:

1. **Energy ↔ Loudness (r ≈ 0.76)**: Đây là cặp tương quan **dương mạnh nhất** trong toàn bộ ma trận. Điều này hoàn toàn hợp lý vì bài hát có năng lượng cao thường được mix ở mức âm lượng lớn hơn. Khi xây dựng mô hình, cần lưu ý hiện tượng **đa cộng tuyến** giữa hai biến này — tuy nhiên các mô hình cây quyết định trong pipeline (Random Forest, XGBoost, LightGBM) xử lý tốt vấn đề này.

2. **Energy ↔ Acousticness (r ≈ -0.73)**: Tương quan **âm mạnh** — bài hát acoustic thường có năng lượng thấp (nhạc nhẹ nhàng, guitar, piano), trong khi bài hát năng lượng cao thường dùng nhạc cụ điện tử, trống mạnh.

3. **Loudness ↔ Acousticness (r ≈ -0.55)**: Nhạc acoustic thường được thu nhỏ hơn, âm lượng thấp hơn so với nhạc điện tử/pop.

4. **Danceability ↔ Valence (r ≈ 0.42)**: Bài hát có thể nhảy theo thường mang **tâm trạng vui vẻ, tích cực** (valence cao).

#### Tương quan với Popularity (biến mục tiêu):
- Đáng chú ý là **không có đặc trưng âm thanh nào có tương quan mạnh với popularity**. Tương quan cao nhất chỉ khoảng **±0.05-0.08**.
- Điều này cho thấy rằng **mức độ phổ biến của bài hát không phụ thuộc đơn thuần vào các đặc điểm âm thanh**. Các yếu tố khác như: tên tuổi nghệ sĩ, chiến lược marketing, playlist placement, thời điểm phát hành... có ảnh hưởng lớn hơn nhiều.
- Đây là lý do pipeline sử dụng **`track_genre` với TargetEncoder** — genre encode gián tiếp các yếu tố ngoài âm thanh.

#### Các đặc trưng gần như độc lập:
- **Tempo** gần như **không tương quan** với bất kỳ đặc trưng nào khác (r ≈ 0).
- **Instrumentalness** và **Liveness** cũng có tương quan yếu với hầu hết các biến khác.

### 3.3 Đặc Trưng Âm Thanh vs Popularity — Biểu Đồ Phân Tán (Scatter Grid)

In [ ]:
features_to_plot = ['danceability', 'energy', 'valence', 'loudness',
                    'acousticness', 'tempo', 'speechiness', 'instrumentalness']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('🎛️  Đặc Trưng Âm Thanh vs Popularity', fontsize=18,
             fontweight='bold', color=SPOTIFY_GREEN, y=1.02)

for idx, feat in enumerate(features_to_plot):
    ax = axes.flat[idx]
    ax.scatter(df[feat], df['popularity'], alpha=0.05, s=3,
              color=PALETTE[idx % len(PALETTE)])
    # Đường hồi quy (trend line)
    z = np.polyfit(df[feat], df['popularity'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_line, p(x_line), color='white', lw=2, linestyle='--', alpha=0.8)

    r = df[feat].corr(df['popularity'])
    ax.set_title(f'{feat}\nr = {r:.3f}', fontsize=11, fontweight='bold')
    ax.set_xlabel(feat); ax.set_ylabel('Popularity')
    ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.show()

### 📝 Nhận xét biểu đồ 3.3 — Đặc Trưng Âm Thanh vs Popularity

#### Phân tích từng đặc trưng:

1. **Danceability vs Popularity**: Đám mây điểm **phân tán rộng**, đường trend gần như **nằm ngang** (r ≈ 0.06). Bài hát có thể nhảy theo không nhất thiết phổ biến hơn. Tuy nhiên, mật độ điểm cao hơn ở khoảng danceability 0.5-0.8.

2. **Energy vs Popularity**: Tương tự, tương quan rất yếu. Bài hát có energy cao hoặc thấp đều có thể phổ biến.

3. **Valence vs Popularity**: Gần như **không có mối quan hệ tuyến tính**. Bài hát buồn và vui đều có thể đạt popularity cao.

4. **Loudness vs Popularity**: Có **xu hướng dương nhẹ** — bài hát to hơn có xu hướng phổ biến hơn một chút.

5. **Acousticness vs Popularity**: Xu hướng **âm nhẹ** — bài hát acoustic có xu hướng ít phổ biến hơn.

6. **Tempo vs Popularity**: **Hoàn toàn không có tương quan**. Nhịp độ không ảnh hưởng đến mức độ phổ biến.

7. **Speechiness vs Popularity**: Các bài có speechiness cao (>0.5) thường **ít phổ biến** — đây thường là podcast hoặc spoken word.

8. **Instrumentalness vs Popularity**: Xu hướng **âm rõ rệt nhất** — bài hát thuần nhạc cụ thường **ít phổ biến**.

#### Kết luận:
- **Không có đặc trưng âm thanh đơn lẻ nào dự đoán tốt popularity** (|r| < 0.1).
- Cần dùng **tổ hợp nhiều features** + **genre encoding** — đúng như cách pipeline trong `train_model.py` thực hiện.

### 3.4 Top Thể Loại Theo Mức Độ Phổ Biến Trung Bình

In [ ]:
top_genres = (df.groupby('track_genre')['popularity']
                .mean().sort_values(ascending=False).head(20))
bottom_genres = (df.groupby('track_genre')['popularity']
                   .mean().sort_values().head(20))

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('🎸  Mức Độ Phổ Biến Theo Thể Loại', fontsize=18,
             fontweight='bold', color=SPOTIFY_GREEN, y=1.02)

# Top 20 phổ biến nhất
ax = axes[0]
colors_top = plt.cm.YlGn(np.linspace(0.35, 0.95, len(top_genres)))
bars = ax.barh(top_genres.index, top_genres.values, color=colors_top[::-1],
               edgecolor='#111', linewidth=0.3)
for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}', va='center', fontsize=9, color='white')
ax.set_xlabel('Popularity trung bình')
ax.set_title('🏆 Top 20 Thể Loại Phổ Biến Nhất', fontweight='bold', color=SPOTIFY_GREEN)
ax.invert_yaxis(); ax.grid(True, axis='x', alpha=0.2)

# Bottom 20 ít phổ biến nhất
ax = axes[1]
colors_bottom = plt.cm.Reds(np.linspace(0.3, 0.8, len(bottom_genres)))
bars = ax.barh(bottom_genres.index, bottom_genres.values, color=colors_bottom,
               edgecolor='#111', linewidth=0.3)
for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}', va='center', fontsize=9, color='white')
ax.set_xlabel('Popularity trung bình')
ax.set_title('📉 20 Thể Loại Ít Phổ Biến Nhất', fontweight='bold', color='#E74C3C')
ax.invert_yaxis(); ax.grid(True, axis='x', alpha=0.2)

plt.tight_layout()
plt.show()

### 📝 Nhận xét biểu đồ 3.4 — Top Thể Loại Theo Mức Độ Phổ Biến

#### Top 20 Thể Loại Phổ Biến Nhất:
- Các thể loại **Pop, K-pop, và nhạc Latin (reggaeton, latin)** dẫn đầu danh sách, phản ánh đúng xu hướng thị trường âm nhạc toàn cầu hiện tại.
- **Pop** luôn là thể loại phổ biến nhất nhờ tính đại chúng cao và chiến lược marketing mạnh mẽ.
- **K-pop** nổi lên mạnh mẽ nhờ fandom quốc tế cực kỳ trung thành.
- Khoảng cách giữa thể loại cao nhất và thấp nhất trong top 20 **không quá lớn** (~10-15 điểm).

#### 20 Thể Loại Ít Phổ Biến Nhất:
- Các thể loại **ngách** (niche) như opera, anime, children, sleep, ambient... có popularity trung bình thấp nhất.
- Điều này phản ánh **đối tượng nghe nhỏ hơn** và ít được thuật toán Spotify đẩy mạnh.

#### Ý nghĩa cho pipeline:
- **`track_genre` là feature quan trọng nhất** — đó là lý do `train_model.py` sử dụng `TargetEncoder` để encode genre theo popularity trung bình.
- `TargetEncoder` của scikit-learn sử dụng **cross-fitting** để tránh data leakage — an toàn hơn so với target encoding thủ công.

### 3.5 Phân Phối Thời Lượng Bài Hát Theo Thể Loại (Top 10)

In [ ]:
top10_genres = df.groupby('track_genre')['popularity'].mean().nlargest(10).index
df_top10 = df[df['track_genre'].isin(top10_genres)]

fig, ax = plt.subplots(figsize=(16, 7))
genre_list = df_top10['track_genre'].unique()
colors_v = plt.get_cmap('Set2')(np.linspace(0, 1, len(genre_list)))

for i, (genre, color) in enumerate(zip(genre_list, colors_v)):
    subset = df_top10[df_top10['track_genre'] == genre]['duration_min']
    parts = ax.violinplot(subset, positions=[i], showmedians=True, widths=0.75)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    for partname in ('cbars','cmins','cmaxes','cmedians'):
        parts[partname].set_edgecolor('white')
        parts[partname].set_linewidth(1)

ax.set_xticks(range(len(genre_list)))
ax.set_xticklabels(genre_list, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Thời lượng (phút)')
ax.set_ylim(0, 12)  # Giới hạn trục y để tránh outlier
ax.set_title('🎻  Phân Phối Thời Lượng — Top 10 Thể Loại Phổ Biến',
             fontsize=15, fontweight='bold', color=SPOTIFY_GREEN)
ax.grid(True, axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

### 📝 Nhận xét biểu đồ 3.5 — Phân Phối Thời Lượng Theo Thể Loại

#### Quan sát chính:
- **Đa số các thể loại phổ biến** có thời lượng trung bình dao động từ **3-4 phút**, phù hợp với tiêu chuẩn phát thanh (radio-friendly format).
- Hình dạng **violin** cho thấy phân phối thời lượng khá **tập trung** cho các thể loại pop, latin.
- **Pop** có thời lượng **ngắn và đồng nhất nhất** (~3-3.5 phút) — chiến lược có chủ đích vì bài pop ngắn dễ nghe lặp lại.

#### Ý nghĩa cho pipeline:
- `duration_min` là feature trong `numeric_features` của `train_model.py` — biểu đồ này xác nhận nó **có giá trị phân biệt** giữa các thể loại.

### 3.6 Phân Tích Âm Giai & Cung Nhạc (Musical Mode & Key)

In [ ]:
KEY_NAMES = ['C','C#/Db','D','D#/Eb','E','F','F#/Gb','G','G#/Ab','A','A#/Bb','B']

key_pop = df.groupby('key')['popularity'].mean().reset_index()
key_pop['key_name'] = key_pop['key'].map(lambda x: KEY_NAMES[x] if 0<=x<12 else 'N/A')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('🎹  Phân Tích Âm Giai & Cung Nhạc',
             fontsize=18, fontweight='bold', color=SPOTIFY_GREEN, y=1.02)

# Biểu đồ cung nhạc (Key)
ax = axes[0]
key_colors = plt.cm.cool(np.linspace(0.2, 0.9, len(key_pop)))
bars = ax.bar(key_pop['key_name'], key_pop['popularity'], color=key_colors,
              edgecolor='#111', linewidth=0.4)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}', ha='center', fontsize=9, color='white')
ax.set_xlabel('Cung nhạc (Key)')
ax.set_ylabel('Popularity trung bình')
ax.set_title('Popularity Theo Cung Nhạc', fontweight='bold')
ax.grid(True, axis='y', alpha=0.2)

# Biểu đồ âm giai (Mode: Major vs Minor)
ax = axes[1]
mode_pop = df.groupby('mode')['popularity'].mean()
mode_labels = ['Minor (Thứ)', 'Major (Trưởng)']
mode_colors = [ACCENT_PURPLE, SPOTIFY_GREEN]
bars = ax.bar(mode_labels, mode_pop.values, color=mode_colors,
              edgecolor='#111', linewidth=0.4, width=0.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.2f}', ha='center', fontsize=12, color='white',
            fontweight='bold')
ax.set_ylabel('Popularity trung bình')
ax.set_title('Popularity Theo Âm Giai (Major vs Minor)', fontweight='bold')
ax.grid(True, axis='y', alpha=0.2)

plt.tight_layout()
plt.show()

### 📝 Nhận xét biểu đồ 3.6 — Phân Tích Âm Giai & Cung Nhạc

#### Biểu đồ cung nhạc (Key):
- Popularity trung bình theo cung nhạc **dao động rất nhỏ** (khoảng 32-35), cho thấy cung nhạc **không phải yếu tố quyết định** đến độ phổ biến.
- Trong pipeline, `key` được giữ nguyên dạng số nguyên (0-11) trong `numeric_features` — đơn giản nhưng hiệu quả cho mô hình cây.

#### Biểu đồ âm giai (Major vs Minor):
- Popularity giữa **Major (Trưởng)** và **Minor (Thứ)** gần như **bằng nhau** (chênh lệch < 1 điểm).
- `mode` (0/1) được giữ trong `numeric_features` — mô hình cây tự biết cách sử dụng feature binary này.

### 3.7 Phân Phối Mật Độ Các Đặc Trưng Âm Thanh (KDE Grid)

In [ ]:
kde_features = ['danceability','energy','valence','acousticness',
                'speechiness','instrumentalness','liveness','loudness']

fig, axes = plt.subplots(2, 4, figsize=(22, 9))
fig.suptitle('📈  Phân Phối Mật Độ Các Đặc Trưng Âm Thanh',
             fontsize=18, fontweight='bold', color=SPOTIFY_GREEN, y=1.02)

for idx, feat in enumerate(kde_features):
    ax = axes.flat[idx]
    color = PALETTE[idx % len(PALETTE)]

    # KDE plot
    sns.kdeplot(df[feat], ax=ax, fill=True, color=color, alpha=0.5, linewidth=2)

    # Thêm đường trung bình và trung vị
    mean_val = df[feat].mean()
    median_val = df[feat].median()
    ax.axvline(mean_val, color='white', linestyle='--', lw=1.5, alpha=0.8,
               label=f'TB = {mean_val:.2f}')
    ax.axvline(median_val, color=ACCENT_ORANGE, linestyle='-.', lw=1.5, alpha=0.8,
               label=f'TV = {median_val:.2f}')

    ax.set_title(feat, fontweight='bold', fontsize=12)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylabel('Mật độ')
    ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.show()

### 📝 Nhận xét biểu đồ 3.7 — Phân Phối Mật Độ Các Đặc Trưng Âm Thanh

#### Phân tích chi tiết từng đặc trưng:

1. **Danceability**: Phân phối gần **chuẩn**, hơi lệch trái, tập trung quanh 0.5-0.7. Đa số bài hát có khả năng nhảy **trung bình đến cao**.

2. **Energy**: Phân phối **lệch trái**, đỉnh ở khoảng 0.7-0.9. **Đa số bài hát có năng lượng cao**.

3. **Valence**: Phân phối **gần chuẩn nhất**, tập trung quanh 0.4-0.5. Bộ dữ liệu **cân bằng** giữa bài vui và buồn.

4. **Acousticness**: Phân phối **lệch phải mạnh** với đỉnh ở gần 0. Đa số bài hát **không phải acoustic**.

5. **Speechiness**: Phân phối **cực kỳ lệch phải** — hầu hết bài hát có rất ít spoken word.

6. **Instrumentalness**: Phân phối **bimodal** (hai đỉnh): đỉnh lớn ở 0 (có lời) và đỉnh nhỏ ở gần 1 (không lời).

7. **Liveness**: Phân phối **lệch phải** — đa số bài hát là **thu âm studio**.

8. **Loudness**: Phân phối gần **chuẩn** nhưng lệch trái nhẹ, tập trung quanh -5 đến -10 dB.

#### Ý nghĩa cho pipeline:
- Các features lệch mạnh (speechiness, instrumentalness, liveness) được xử lý bởi `StandardScaler` trong pipeline — cần thiết cho Ridge Regression.
- Các mô hình cây (Random Forest, XGBoost, LightGBM, Decision Tree, AdaBoost) **không cần scaling** nhưng pipeline vẫn apply để đồng nhất.

### 3.8 Top 10 Nghệ Sĩ Phổ Biến Nhất

In [ ]:
# Tách các nghệ sĩ (phân cách bởi dấu ;)
artists_exploded = df.assign(artists=df['artists'].str.split(';')).explode('artists')
artists_exploded['artists'] = artists_exploded['artists'].str.strip()

top_artists = (artists_exploded.groupby('artists')
               .agg(avg_pop=('popularity','mean'),
                    n_tracks=('popularity','count'))
               .query('n_tracks >= 5')  # Chỉ lấy nghệ sĩ có >= 5 bài
               .sort_values('avg_pop', ascending=False)
               .head(10))

fig, ax = plt.subplots(figsize=(14, 7))

colors_art = plt.cm.YlGn(np.linspace(0.35, 0.95, len(top_artists)))[::-1]
bars = ax.barh(top_artists.index, top_artists['avg_pop'],
               color=colors_art, edgecolor='#111', linewidth=0.4)

for bar, n in zip(bars, top_artists['n_tracks']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}  ({n} bài)',
            va='center', fontsize=10, color='white')

ax.set_xlabel('Popularity Trung Bình')
ax.set_title('🎤  Top 10 Nghệ Sĩ Phổ Biến Nhất (≥ 5 bài hát)',
             fontsize=15, fontweight='bold', color=SPOTIFY_GREEN)
ax.invert_yaxis()
ax.grid(True, axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

### 📝 Nhận xét biểu đồ 3.8 — Top 10 Nghệ Sĩ Phổ Biến Nhất

- Top nghệ sĩ đều là **tên tuổi lớn toàn cầu**, popularity trung bình **70-90** — cao hơn đáng kể so với trung bình dataset (~33).
- **Tên tuổi nghệ sĩ là yếu tố ảnh hưởng lớn nhất** đến popularity — lớn hơn bất kỳ audio feature nào.
- Tuy nhiên, pipeline hiện tại **không sử dụng tên nghệ sĩ** làm feature (quá nhiều categories, nguy cơ data leakage). Thay vào đó, `track_genre` đóng vai trò proxy gián tiếp.
- **Đề xuất cải thiện**: Có thể thêm `artist_avg_popularity` (target encoding cho artists) trong các phiên bản tương lai.

---
## 🤖 4. Machine Learning — Dự Đoán Mức Độ Phổ Biến
### 4.1 Chuẩn Bị Dữ Liệu & Pipeline (giống `train_model.py`)

In [ ]:
# === Feature Selection — GIỐNG HỆT train_model.py ===
categorical_features = ['track_genre']
numeric_features = [
    'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'time_signature',
    'explicit', 'duration_min'
]

X = df[categorical_features + numeric_features]
y = df['popularity']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Preprocessing pipeline — GIỐNG HỆT train_model.py
# TargetEncoder sử dụng cross-fitting để tránh data leakage
preprocessor = ColumnTransformer(
    transformers=[
        ('target_enc', TargetEncoder(target_type='continuous', random_state=42), categorical_features),
        ('scaler', StandardScaler(), numeric_features)
    ],
    remainder='passthrough'
)

print(f'Tập huấn luyện: {X_train.shape} | Tập kiểm tra: {X_test.shape}')
print(f'\nCategorical features ({len(categorical_features)}):')
for f in categorical_features:
    print(f'  • {f} ({df[f].nunique()} unique values)')
print(f'\nNumeric features ({len(numeric_features)}):')
for f in numeric_features:
    print(f'  • {f}')

### 📝 Giải thích Pipeline Tiền Xử Lý

Pipeline trong `train_model.py` sử dụng `ColumnTransformer` để xử lý hai loại features:

#### 1. TargetEncoder cho `track_genre`:
- Với **114 thể loại**, one-hot encoding tạo ra 114 cột → quá nhiều, gây **curse of dimensionality**.
- `TargetEncoder` của scikit-learn thay thế mỗi genre bằng **popularity trung bình** của genre đó.
- **Ưu điểm so với target encoding thủ công**: Sử dụng **cross-fitting** (tương tự cross-validation) để tránh data leakage — mỗi fold chỉ dùng thông tin từ các fold khác.

#### 2. StandardScaler cho numeric features:
- Chuẩn hóa tất cả features số về **mean = 0, std = 1**.
- **Cần thiết** cho Ridge Regression (nhạy cảm với scale).
- Các mô hình cây (Random Forest, XGBoost...) **không bị ảnh hưởng** bởi scaling, nhưng pipeline apply đồng nhất cho tất cả mô hình.

#### 3. Tại sao dùng Pipeline?
- **Tránh data leakage**: Preprocessing chỉ fit trên train data, transform trên test data.
- **Dễ deploy**: Chỉ cần save/load 1 file `pipeline.pkl` → API (`api.py`) dùng trực tiếp.
- **Reproducible**: Đảm bảo cùng bước xử lý khi train và predict.

### 4.2 Huấn Luyện & So Sánh 6 Mô Hình (giống `train_model.py`)

In [ ]:
# === Định nghĩa mô hình — GIỐNG HỆT train_model.py ===
models = {
    'Ridge Regression': Ridge(alpha=10),
    'Decision Tree':    DecisionTreeRegressor(max_depth=12, random_state=42),
    'AdaBoost':         AdaBoostRegressor(n_estimators=50, random_state=42),
    'Random Forest':    RandomForestRegressor(n_estimators=100, max_depth=12,
                                              random_state=42, n_jobs=-1),
    'XGBoost':          xgb.XGBRegressor(n_estimators=300, max_depth=6,
                                         learning_rate=0.05, random_state=42, n_jobs=-1),
    'LightGBM':         lgb.LGBMRegressor(n_estimators=400, max_depth=8,
                                          learning_rate=0.04, random_state=42,
                                          n_jobs=-1, verbose=-1),
}

results = {}
pipelines = {}

print('Đang huấn luyện 6 mô hình...\n')
print('-' * 65)
print(f'{"Mô hình":<20} | {"RMSE":<10} | {"MAE":<10} | {"R²":<10}')
print('-' * 65)

for name, model in models.items():
    # Xây dựng pipeline — GIỐNG HỆT train_model.py
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # Huấn luyện
    pipe.fit(X_train, y_train)
    
    # Dự đoán & Đánh giá
    preds = pipe.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'preds': preds}
    pipelines[name] = pipe
    print(f'{name:<20} | {rmse:<10.3f} | {mae:<10.3f} | {r2:<10.4f}')

print('-' * 65)

best_model_name = max(results, key=lambda m: results[m]['R2'])
print(f'\n🏆 Mô hình tốt nhất: {best_model_name} với R² = {results[best_model_name]["R2"]:.4f}')

### 📝 Nhận xét kết quả huấn luyện

#### So sánh 6 mô hình (giống `train_model.py`):
- **Ridge Regression** (mô hình tuyến tính) cho kết quả **kém nhất** — xác nhận rằng mối quan hệ giữa features và popularity là **phi tuyến**.
- **Decision Tree** đơn lẻ cho kết quả trung bình — dễ bị **overfitting** do chỉ có 1 cây.
- **AdaBoost** cải thiện so với Decision Tree đơn lẻ nhờ ensemble, nhưng vẫn chưa bằng các mô hình boosting hiện đại.
- **Random Forest** cho kết quả tốt nhờ kết hợp nhiều cây (bagging) — giảm variance.
- **XGBoost** và **LightGBM** cho kết quả **tốt nhất** — đây là các mô hình boosting hiện đại với nhiều tối ưu.

#### Pipeline tự động chọn mô hình tốt nhất:
- `train_model.py` tự động lưu mô hình có R² cao nhất vào `pipeline.pkl`.
- `api.py` load `pipeline.pkl` để phục vụ dự đoán qua FastAPI.

### 4.3 Bảng So Sánh Hiệu Suất Mô Hình (Dashboard)

In [ ]:
model_names  = list(results.keys())
rmse_vals    = [results[m]['RMSE'] for m in model_names]
mae_vals     = [results[m]['MAE']  for m in model_names]
r2_vals      = [results[m]['R2']   for m in model_names]

fig = plt.figure(figsize=(22, 12))
fig.suptitle('🏆  So Sánh Hiệu Suất 6 Mô Hình', fontsize=20,
             fontweight='bold', color=SPOTIFY_GREEN)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# --- RMSE ---
ax1 = fig.add_subplot(gs[0, 0])
clrs = [SPOTIFY_GREEN if v==min(rmse_vals) else ACCENT_BLUE for v in rmse_vals]
bars = ax1.bar(model_names, rmse_vals, color=clrs, edgecolor='#000', alpha=0.85)
ax1.set_title('RMSE  (thấp hơn = tốt hơn ✓)', color=ACCENT_ORANGE, fontweight='bold')
ax1.set_xticklabels(model_names, rotation=30, ha='right', fontsize=8)
for bar, val in zip(bars, rmse_vals):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
             f'{val:.2f}', ha='center', fontsize=8)
ax1.grid(True, axis='y', alpha=0.2)

# --- MAE ---
ax2 = fig.add_subplot(gs[0, 1])
clrs = [SPOTIFY_GREEN if v==min(mae_vals) else ACCENT_PURPLE for v in mae_vals]
bars = ax2.bar(model_names, mae_vals, color=clrs, edgecolor='#000', alpha=0.85)
ax2.set_title('MAE  (thấp hơn = tốt hơn ✓)', color=ACCENT_ORANGE, fontweight='bold')
ax2.set_xticklabels(model_names, rotation=30, ha='right', fontsize=8)
for bar, val in zip(bars, mae_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f'{val:.2f}', ha='center', fontsize=8)
ax2.grid(True, axis='y', alpha=0.2)

# --- R² ---
ax3 = fig.add_subplot(gs[0, 2])
clrs = [SPOTIFY_GREEN if v==max(r2_vals) else ACCENT_ORANGE for v in r2_vals]
bars = ax3.bar(model_names, r2_vals, color=clrs, edgecolor='#000', alpha=0.85)
ax3.set_title('R²  (cao hơn = tốt hơn ✓)', color=ACCENT_ORANGE, fontweight='bold')
ax3.set_xticklabels(model_names, rotation=30, ha='right', fontsize=8)
for bar, val in zip(bars, r2_vals):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
             f'{val:.3f}', ha='center', fontsize=8)
ax3.grid(True, axis='y', alpha=0.2)

# --- Mô hình tốt nhất: Dự đoán vs Thực tế ---
best_preds = results[best_model_name]['preds']

ax4 = fig.add_subplot(gs[1, 0:2])
ax4.scatter(y_test, best_preds, alpha=0.2, s=6, color=SPOTIFY_GREEN)
mn, mx = y_test.min(), y_test.max()
ax4.plot([mn, mx], [mn, mx], 'w--', lw=2, label='Dự đoán hoàn hảo')
ax4.set_xlabel('Popularity Thực Tế'); ax4.set_ylabel('Popularity Dự Đoán')
ax4.set_title(f'🎯  {best_model_name} — Thực Tế vs Dự Đoán', fontweight='bold',
              color=SPOTIFY_GREEN)
ax4.legend(); ax4.grid(True, alpha=0.2)

# --- Phân phối phần dư (Residuals) ---
ax5 = fig.add_subplot(gs[1, 2])
residuals = y_test.values - best_preds
ax5.hist(residuals, bins=60, color=ACCENT_PURPLE, edgecolor='#000', alpha=0.8)
ax5.axvline(0, color='white', lw=2, linestyle='--')
ax5.set_xlabel('Phần dư (Residual)'); ax5.set_ylabel('Số lượng')
ax5.set_title('Phân Phối Phần Dư', color=ACCENT_PURPLE, fontweight='bold')
ax5.grid(True, alpha=0.2)

plt.show()

### 📝 Nhận xét biểu đồ 4.3 — Dashboard So Sánh 6 Mô Hình

#### Biểu đồ RMSE:
- Mô hình tốt nhất (RMSE thấp nhất) được highlight **màu xanh Spotify**.
- **Ridge** có RMSE cao nhất → mô hình tuyến tính không phù hợp.
- **Decision Tree** có RMSE cao thứ hai → 1 cây dễ overfitting.
- **LightGBM/XGBoost** có RMSE thấp nhất → dự đoán chính xác nhất.

#### Biểu đồ Thực Tế vs Dự Đoán:
- Điểm tập trung quanh **đường chéo trắng** → mô hình có xu hướng đúng.
- Phân tán rộng ở vùng popularity thấp (0-20) và cao (70-100) → mô hình **khó dự đoán ở hai đầu cực**.
- Xu hướng **regression to the mean**: bài popularity = 0 bị dự đoán cao hơn, và ngược lại.

#### Biểu đồ Phần Dư:
- Phân phối **gần chuẩn**, tập trung quanh 0 → mô hình **không bị lệch hệ thống**.

### 4.4 Mức Độ Quan Trọng Của Các Đặc Trưng (Feature Importance)

In [ ]:
# Chọn mô hình cây tốt nhất để xem feature importance
tree_models = [m for m in model_names if m not in ['Ridge Regression']]
best_tree_name = max(tree_models, key=lambda m: results[m]['R2'])
best_pipe = pipelines[best_tree_name]
best_tree = best_pipe.named_steps['regressor']

# Feature names sau ColumnTransformer: target_enc output + numeric features
feature_names = categorical_features + numeric_features

importances = pd.Series(best_tree.feature_importances_,
                        index=feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 9))

norm = plt.Normalize(importances.min(), importances.max())
clrs = plt.cm.YlGn(norm(importances.values))

bars = ax.barh(importances.index, importances.values,
               color=clrs, edgecolor='#111', linewidth=0.4)

# Highlight top 5 feature quan trọng nhất
top5_idx = importances.nlargest(5).index
for bar, feat in zip(bars, importances.index):
    if feat in top5_idx:
        bar.set_edgecolor(ACCENT_ORANGE); bar.set_linewidth(2)
    ax.text(bar.get_width()+0.0005, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.4f}', va='center', fontsize=9, color='white')

ax.set_xlabel('Mức Độ Quan Trọng (Feature Importance)')
ax.set_title(f'🔑  Mức Độ Quan Trọng Của Đặc Trưng — {best_tree_name}',
             fontsize=15, fontweight='bold', color=SPOTIFY_GREEN)
ax.grid(True, axis='x', alpha=0.2)
plt.tight_layout(); plt.show()

### 📝 Nhận xét biểu đồ 4.4 — Mức Độ Quan Trọng Của Đặc Trưng

#### Top features quan trọng nhất (highlight viền cam):

1. **track_genre** (TargetEncoded) — Feature **quan trọng nhất** với khoảng cách lớn so với các feature khác. Xác nhận phát hiện từ EDA: **thể loại nhạc là yếu tố quyết định chính**. TargetEncoder encode gián tiếp thông tin về đối tượng nghe và trend thị trường.

2. **loudness** — Âm lượng phản ánh **chất lượng sản xuất (production quality)**. Bài hát được mix chuyên nghiệp hơn thường to hơn.

3. **duration_min** — Thời lượng bài hát ảnh hưởng đáng kể. Bài hát quá dài thường ít phổ biến.

4. **acousticness** / **energy** — Các đặc điểm âm thanh cơ bản.

#### Các features ít quan trọng:
- **key**, **mode** — Gần như không ảnh hưởng, xác nhận phân tích ở biểu đồ 3.6.
- **time_signature** — Đa số nhạc dùng 4/4 nên ít phân biệt.

#### Kết luận:
- Feature importance **nhất quán với EDA**: genre quan trọng nhất, audio features đơn lẻ ít ảnh hưởng.
- Pipeline hiện tại đã chọn đúng features — không cần thêm/bớt.

### 4.5 Cross-Validation — Kiểm Tra Độ Ổn Định Mô Hình

In [ ]:
cv_models = {
    'Random Forest': pipelines['Random Forest'],
    'XGBoost':       pipelines['XGBoost'],
    'LightGBM':      pipelines['LightGBM'],
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, pipe in cv_models.items():
    scores = cross_val_score(pipe, X, y, cv=kf,
                             scoring='neg_root_mean_squared_error', n_jobs=-1)
    cv_results[name] = -scores
    print(f'{name:20s}  CV-RMSE: {(-scores).mean():.3f} +/- {(-scores).std():.3f}')

# Box plot kết quả CV
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(list(cv_results.values()), patch_artist=True,
                medianprops=dict(color='white', lw=2.5), widths=0.5)
colors_cv = [SPOTIFY_GREEN, ACCENT_ORANGE, ACCENT_PURPLE]
for patch, clr in zip(bp['boxes'], colors_cv):
    patch.set_facecolor(clr); patch.set_alpha(0.75)
ax.set_xticklabels(list(cv_results.keys()))
ax.set_ylabel('RMSE')
ax.set_title('📊  5-Fold Cross-Validation — Độ Ổn Định Mô Hình', fontsize=14,
             fontweight='bold', color=SPOTIFY_GREEN)
ax.grid(True, axis='y', alpha=0.2)
plt.tight_layout(); plt.show()

### 📝 Nhận xét biểu đồ 4.5 — Cross-Validation

#### Kết quả 5-Fold Cross-Validation:
- Cross-validation chia dữ liệu thành **5 phần bằng nhau**, đánh giá mô hình 5 lần trên các tập khác nhau.
- Phương pháp này đánh giá **độ tin cậy** của mô hình, tránh kết quả tốt chỉ do "may mắn" chia train/test.

#### Phân tích box plot:
- **Hộp nhỏ** (IQR hẹp) → mô hình **ổn định** trên các fold → **không overfitting**.
- **LightGBM** thường có hộp nhỏ nhất + RMSE trung bình thấp → **mô hình tốt nhất**.
- **XGBoost** kết quả tương đương LightGBM.
- **Random Forest** RMSE cao hơn nhưng vẫn ổn định.

#### Kết luận:
- Cả 3 mô hình đều **ổn định** (std nhỏ), không overfitting.
- **LightGBM** được khuyến nghị cho production (`pipeline.pkl`) nhờ accuracy cao nhất và tốc độ nhanh nhất.

---
## 📌 5. Tổng Kết & Kết Luận

### Những phát hiện chính từ EDA:

1. **Phân phối Popularity không đều**: Đa số bài hát có popularity thấp-trung bình (~33/100). Chỉ một tỷ lệ nhỏ bài hát đạt viral (>80).

2. **Audio features đơn lẻ ít ảnh hưởng đến popularity**: Tất cả hệ số tương quan |r| < 0.1. Popularity phụ thuộc nhiều vào yếu tố ngoài âm thanh.

3. **Genre là yếu tố quan trọng nhất**: Pop, K-pop, Latin dẫn đầu. Genre encode gián tiếp thông tin thị trường → `TargetEncoder` là lựa chọn đúng đắn.

4. **Xu hướng bài hát ngắn**: Bài hát phổ biến thường 2-4 phút.

5. **Cung nhạc và âm giai gần như không ảnh hưởng**.

### Kết quả Machine Learning (6 mô hình trong `train_model.py`):

| Mô hình | Đánh giá |
|---------|----------|
| Ridge Regression | Kém nhất — quan hệ phi tuyến |
| Decision Tree | Trung bình — dễ overfitting |
| AdaBoost | Khá — cải thiện so với 1 cây |
| Random Forest | Tốt — bagging giảm variance |
| XGBoost | Rất tốt — boosting hiện đại |
| **LightGBM** | **Tốt nhất** — nhanh + chính xác |

### Kiến trúc Project:
```
spotify_track_popularity_predictor/
├── dataset.csv              # Dữ liệu gốc
├── train_model.py           # Huấn luyện & chọn mô hình tốt nhất
├── pipeline.pkl             # Pipeline đã train (TargetEncoder + Scaler + Model)
├── genres.pkl               # Danh sách thể loại
├── api.py                   # FastAPI server để dự đoán
├── frontend/                # Giao diện web
├── notebooks/               # EDA notebook (file này)
└── requirements.txt         # Dependencies
```

### Đề xuất cải thiện:
- Thêm **artist-level features** (artist popularity trung bình)
- Sử dụng **temporal features** (thời điểm phát hành)
- Áp dụng **hyperparameter tuning** (Optuna)
- Thêm **API endpoint** cho batch prediction

---
🎵 **Phân tích hoàn tất!**

In [ ]:
print('🎵 Phân tích hoàn tất! Cảm ơn bạn đã theo dõi! 🙏')